# Video Deepfake Detection with Efficient ViT — Inference Notebook with Gradio

**Derived from** [davide-coccomini/Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection](https://github.com/davide-coccomini/Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection),
the official implementation of:

> Coccomini, D., Caldelli, R., Falchi, F., & Amato, G. (2021).
> *Combining EfficientNet and Vision Transformers for Video Deepfake Detection.*
> ICIAP 2021. ([arXiv:2107.02612](https://arxiv.org/abs/2107.02612))

**What this notebook does.** A self-contained, environment-agnostic (CPU/GPU auto-detected)
inference demo of the paper's **Efficient ViT** architecture, wrapped in a Gradio web interface:

- The model extracts features with a frozen-ish **EfficientNet-B0** backbone (only the last three
  MBConv blocks were fine-tuned in training), treats the resulting 7x7x1280 feature map as patches,
  and classifies them with a small **Vision Transformer**.
- Per-frame predictions are aggregated (mean of frame scores) into a single video-level verdict,
  following the video-level aggregation logic of the original `utils.py` (`custom_video_round`).
- The original training pipeline operates on **face crops** extracted with MTCNN
  (`preprocessing/detect_faces.py`); this notebook therefore applies optional MTCNN face cropping
  before classification to match the model's expected input distribution.

The repository is released under the **MIT License**, so both code and weights can be reused and
modified with attribution.

## 1. Setup

Install the required libraries and clone the original repository, which provides the model definition (`efficient-vit/efficient_vit.py`) and the configuration used for training.

In [ ]:
%pip install -q torch torchvision opencv-python-headless efficientnet_pytorch einops pyyaml facenet-pytorch gradio

In [ ]:
import os

REPO_URL = ("https://github.com/davide-coccomini/"
            "Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection.git")
REPO_DIR = "Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone --depth 1 {REPO_URL}")

import sys
sys.path.insert(0, os.path.join(os.getcwd(), REPO_DIR, "efficient-vit"))
print("Repository ready.")

## 2. Imports and environment detection

The original `test.py` assumes CUDA (`model.cuda()`). This notebook is environment-agnostic:
it selects CUDA when available and falls back to CPU otherwise.

In [ ]:
import sys
import types
import urllib.request

import cv2
import numpy as np
import torch
import yaml

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("No GPU detected; inference will work but run noticeably slower.")

# The original efficient_vit.py does 'from utils import resize', where utils.py pulls in
# albumentations. To keep the dependency footprint small, we provide a minimal shim module
# that implements the same resize helper, so the original model file imports unchanged.
_utils = types.ModuleType("utils")
def resize(image, image_size):
    try:
        return cv2.resize(image, dsize=(image_size, image_size))
    except Exception:
        return []
_utils.resize = resize
if "utils" not in sys.modules:
    sys.modules["utils"] = _utils

from efficient_vit import EfficientViT  # original model definition, unmodified
print("Model definition imported from the original repository.")

## 3. Load the pretrained Efficient ViT model

The authors distribute the pretrained checkpoint (`efficient_vit.pth`, EfficientNet-B0 variant)
from their institute server, with a Google Drive folder as fallback (see the repository README).
This notebook tries the direct link first and falls back to clear instructions if it is offline.

In [ ]:
MODELS_DIR = "pretrained_models"
os.makedirs(MODELS_DIR, exist_ok=True)
CKPT_PATH = os.path.join(MODELS_DIR, "efficient_vit.pth")

WEIGHTS_URL = "http://datino.isti.cnr.it/efficientvit_deepfake/efficient_vit.pth"
GDRIVE_FALLBACK = ("https://drive.google.com/drive/folders/"
                   "19bNOs8_rZ7LmPP3boDS3XvZcR1iryHR1")

if not os.path.exists(CKPT_PATH):
    try:
        urllib.request.urlretrieve(WEIGHTS_URL, CKPT_PATH)
        print("Weights downloaded from the authors' server.")
    except Exception as e:
        raise RuntimeError(
            f"The authors' server is unreachable ({e}).\n"
            f"Please download 'efficient_vit.pth' manually from:\n  {GDRIVE_FALLBACK}\n"
            f"and place it at: {CKPT_PATH}"
        )

print(f"Checkpoint ready: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)} bytes)")

In [ ]:
# Use the architecture.yaml shipped with the repository (the configuration the
# checkpoint was trained with). EfficientNet-B0 variant -> channels = 1280.
CONFIG_PATH = os.path.join(REPO_DIR, "efficient-vit", "configs", "architecture.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
print("Model configuration loaded from architecture.yaml")

model = EfficientViT(config=config, channels=1280, selected_efficient_net=0)
state_dict = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE).eval()
print("Efficient ViT loaded and set to evaluation mode.")

## 4. Preprocessing

The original pipeline (in `utils.py::transform_frame` and `transforms/albu.py`) resizes each face
crop isotropically so that its larger side equals the model's input size, then pads to a square
with replicated borders. The input size is `model.image-size = 224` per `architecture.yaml`.

Because the model was trained on face crops, this notebook applies MTCNN face detection
(the same detector family used by the authors' preprocessing, which is based on
Selim Seferbekov's DFDC pipeline) and classifies the largest detected face; if no face is found,
the full frame is classified and the verdict is flagged as lower-confidence.

In [ ]:
from facenet_pytorch import MTCNN

IMAGE_SIZE = config["model"]["image-size"]
mtcnn = MTCNN(device=DEVICE, keep_all=True, post_process=False)

def isotropic_resize(img, size, interpolation_down=cv2.INTER_AREA, interpolation_up=cv2.INTER_CUBIC):
    """Faithful copy of transforms/albu.py::isotropically_resize_image."""
    h, w = img.shape[:2]
    if max(w, h) == size:
        return img
    if w > h:
        scale = size / w; h = int(h * scale); w = size
    else:
        scale = size / h; w = int(w * scale); h = size
    interpolation = interpolation_up if scale > 1 else interpolation_down
    return cv2.resize(img.astype("uint8"), (w, h), interpolation=interpolation)

def transform_frame(image):
    """Faithful copy of utils.py::transform_frame (isotropic resize + border-replicate padding)."""
    img = isotropic_resize(image, IMAGE_SIZE,
                           interpolation_down=cv2.INTER_LINEAR,
                           interpolation_up=cv2.INTER_LINEAR)
    h, w = img.shape[:2]
    if h < IMAGE_SIZE or w < IMAGE_SIZE:  # PadIfNeeded, border_mode=cv2.BORDER_REPLICATE
        pad_h = max(0, IMAGE_SIZE - h); pad_w = max(0, IMAGE_SIZE - w)
        img = cv2.copyMakeBorder(img, pad_h // 2, pad_h - pad_h // 2,
                                 pad_w // 2, pad_w - pad_w // 2,
                                 cv2.BORDER_REPLICATE)
    return img

def extract_face(image_rgb):
    """Return the largest detected face crop, or the full frame if no face is found."""
    result = mtcnn.detect(image_rgb)
    boxes, probs = result[0], result[1] if len(result) > 1 else None
    if boxes is None:
        return image_rgb, False
    areas = [(b[2] - b[0]) * (b[3] - b[1]) for b in boxes]
    x1, y1, x2, y2 = [int(v) for v in boxes[int(np.argmax(areas))]]
    x1, y1 = max(0, x1), max(0, y1)
    x2 = min(image_rgb.shape[1], x2); y2 = min(image_rgb.shape[0], y2)
    if x2 - x1 < 10 or y2 - y1 < 10:
        return image_rgb, False
    return image_rgb[y1:y2, x1:x2], True

def frames_to_tensor(frames_rgb):
    """Apply the original preprocessing to a list of RGB numpy frames."""
    batch = [transform_frame(f).astype("float32") / 255.0 for f in frames_rgb]
    batch = [np.transpose(f, (2, 0, 1)) for f in batch]  # HWC -> CHW
    return torch.from_numpy(np.stack(batch)).float().to(DEVICE)

## 5. Inference

`utils.py::check_correct` applies a sigmoid to the raw model output: the score is the probability
that a frame is fake. Video-level aggregation follows `custom_video_round`: frames are sampled
equidistantly (`frames_per_video = 30` in the config), classified individually, and the video
score is the mean of the frame scores. A video is predicted FAKE when its mean score exceeds 0.5.

In [ ]:
FRAMES_PER_VIDEO = 30  # from architecture.yaml: frames-per-video

@torch.no_grad()
def predict_video(video_path):
    """Sample equidistant frames, classify each face crop, aggregate to a video score."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // FRAMES_PER_VIDEO)

    frames_rgb, face_found = [], False
    index = 0
    while len(frames_rgb) < FRAMES_PER_VIDEO:
        cap.set(cv2.CAP_PROP_POS_FRAMES, index * step)
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        crop, found = extract_face(rgb)
        face_found = face_found or found
        frames_rgb.append(crop)
        index += 1
    cap.release()

    if not frames_rgb:
        return None, None, []

    logits = model(frames_to_tensor(frames_rgb)).flatten()
    frame_scores = logits.sigmoid().cpu().numpy().tolist()
    video_score = float(np.mean(frame_scores))
    return video_score, face_found, frame_scores

@torch.no_grad()
def predict_image(image_rgb):
    """Classify a single image (face crop if detected, otherwise the full frame)."""
    crop, found = extract_face(image_rgb)
    processed = transform_frame(crop)
    x = torch.from_numpy(
        np.transpose(processed.astype("float32") / 255.0, (2, 0, 1))
    ).unsqueeze(0).float().to(DEVICE)
    score = float(model(x).flatten().sigmoid().cpu())
    return score, found

## 6. Gradio web interface

The interface provides three tabs: single image (upload or webcam), and video upload with a
per-frame score chart.

Launch options:

- `SHARE = True`: Gradio creates a temporary public URL (hosted tunnel, valid ~72 hours),
  accessible from any device on the web. Useful on Colab or machines behind NAT.
- `SHARE = False`: the interface is served locally at the printed URL, reachable only from the
  machine running this notebook.

For a permanent deployment, run this notebook on a server with a public IP or reverse proxy and
use `demo.launch(server_name="0.0.0.0", server_port=7860)`.

In [ ]:
import gradio as gr
import pandas as pd

def classify_image_ui(image):
    if image is None:
        return {"REAL": 0.0, "FAKE": 0.0}, ""
    score, face_found = predict_image(image)
    note = "" if face_found else "No face detected; the full frame was classified. "
    note += ("Scores near 0.5 indicate uncertainty. "
             "The model was trained on DFDC-style face crops.")
    return {"REAL": 1.0 - score, "FAKE": score}, note

def classify_video_ui(video_path):
    if video_path is None:
        return {"REAL": 0.0, "FAKE": 0.0}, None, ""
    video_score, face_found, frame_scores = predict_video(video_path)
    if video_score is None:
        return {"REAL": 0.0, "FAKE": 0.0}, None, "Could not read any frames from the video."
    df = pd.DataFrame({
        "frame": list(range(1, len(frame_scores) + 1)),
        "p_fake": frame_scores,
    })
    note = "" if face_found else "No face detected in any frame; full frames were classified. "
    note += f"Video score = mean of {len(frame_scores)} frame scores (threshold 0.5)."
    return {"REAL": 1.0 - video_score, "FAKE": video_score}, df, note

with gr.Blocks(title="Video Deepfake Detection — Efficient ViT") as demo:

    gr.Markdown(
        """
        # Video Deepfake Detection — Efficient ViT
        EfficientNet-B0 features + Vision Transformer (Coccomini et al., ICIAP 2021).
        Trained on DFDC; a probability above 0.5 for FAKE means the content is predicted
        to be manipulated.
        """
    )

    with gr.Tab("Video"):
        video_in = gr.Video(label="Input video")
        video_btn = gr.Button("Classify video", variant="primary")
        video_out = gr.Label(label="Verdict (video level)", num_top_classes=2)
        video_chart = gr.LinePlot(
            x="frame", y="p_fake", y_lim=[0, 1],
            title="Per-frame p(fake)", height=300,
        )
        video_note = gr.Markdown()

    with gr.Tab("Image / Webcam"):
        with gr.Row():
            with gr.Column():
                image_in = gr.Image(type="numpy", label="Input image",
                                    sources=["upload", "webcam"])
                image_btn = gr.Button("Classify image", variant="primary")
            with gr.Column():
                image_out = gr.Label(label="Verdict", num_top_classes=2)
                image_note = gr.Markdown()

    gr.Markdown(
        """
        Notes: per-frame scores are aggregated by mean, mirroring the video-level
        aggregation of the original implementation. Face crops (MTCNN) match the
        model's training distribution; verdicts on faceless content are less reliable.
        """
    )

    video_btn.click(classify_video_ui, inputs=video_in,
                    outputs=[video_out, video_chart, video_note])
    image_btn.click(classify_image_ui, inputs=image_in,
                    outputs=[image_out, image_note])

print("Interface defined. Launch in the next cell.")

In [ ]:
# Set SHARE = True to obtain a temporary public URL (valid ~72 hours).
# Set SHARE = False for local access only (http://127.0.0.1:7860).
SHARE = True

demo.queue()
demo.launch(share=SHARE)

## 7. Interpretation and limitations

- The reported value is `p(fake) = sigmoid(model(face_crop))`; per-frame scores are averaged for
  the video-level verdict, following `custom_video_round` in the original `utils.py`.
- The model was trained on the **DFDC** dataset (face-swap manipulations). Performance on other
  manipulation families (Face2Face, FaceShifter, NeuralTextures, ...) is reported in the paper but
  is lower than on DFDC.
- Accuracy is well below 100 percent: this detector is a strong signal, not a definitive verdict.
  Scores near 0.5 should be treated as uncertain.
- Verdicts on frames without a detected face are less reliable, since the model expects face crops.
- The sibling architecture **Cross Efficient ViT** (`cross-efficient-vit/`) can be used analogously:
  download `cross_efficient_vit.pth`, build `CrossEfficientViT` with the cross-efficient-vit config,
  and reuse the same preprocessing. Its model file imports EfficientNet from a local subfolder of
  the repository, so the import path differs slightly from the Efficient ViT variant used here.

## Citation

```bibtex
@inproceedings{coccomini2021combining,
      title={Combining EfficientNet and Vision Transformers for Video Deepfake Detection},
      author={Coccomini, Davide and Caldelli, Roberto and Falchi, Fabrizio and Amato, Giuseppe},
      booktitle={International Conference on Image Analysis and Processing (ICIAP)},
      pages={219--229},
      year={2021},
}
```

## Source attribution

This notebook derives from the MIT-licensed repository
[Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection](https://github.com/davide-coccomini/Combining-EfficientNet-and-Vision-Transformers-for-Video-Deepfake-Detection).
The model definition is imported unmodified from `efficient-vit/efficient_vit.py`; the
preprocessing reproduces `utils.py::transform_frame` and `transforms/albu.py::IsotropicResize`;
the aggregation reproduces `utils.py::custom_video_round`; the configuration is the repository's
`configs/architecture.yaml`.